# NBA Series Predictor
The feaures we are using to predict a series outcomes are:
- ORTG_DIFF (Offensive Rating — Points per 100 posessions )
- DRTG_DIFF (Defensive Rating — Points conceeded per 100 posessions)
- PACE_DIFF (Pace rating - Number of posessions)
- NET_RTG_DIFF (Net rating — )
- W_PCT_DIFF (Win percentage)

The prediction is given our features, do we expect the higher seeded team to win?

We will use a logistic regression to give us a probability on if the favourite team will win against its matchup




In [1]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score, 
                             confusion_matrix, classification_report)

DATA_PATH = Path('data/series_features.csv')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

TRAIN_CUTOFF = '2018-19'

FEATURE_COLS = [
    'ORTG_DIFF',
    'DRTG_DIFF',
    'PACE_DIFF',
    'W_PCT_DIFF',
    'SEED_DIFF'
]

TARGET_COL = 'HIGHER_SEED_WINS'

def season_split(df):
    train = df[df['SEASON'] <= TRAIN_CUTOFF].copy()
    test  = df[df['SEASON'] >  TRAIN_CUTOFF].copy()

    return train,test

def scale(train, test, feature_cols):
    scaler = StandardScaler()

    X_train = scaler.fit_transform(train[feature_cols])
    X_test = scaler.transform(test[feature_cols])

    y_train = train[TARGET_COL].values
    y_test = test[TARGET_COL].values

    return X_train, X_test, y_train, y_test, scaler



In [2]:
def train_model(X_train, y_train):
    model = LogisticRegression(
        C = 1.0,
        max_iter=1000,
        random_state=42,
        class_weight='balanced'
    )

    model.fit(X_train,y_train)

    with open(MODEL_DIR / 'logreg_model.pkl', 'wb') as f:
        pickle.dump(model, f)

    return model

def evaluate(model, X_train, X_test, y_train, y_test,feature_cols, train_df, test_df):
    train_preds = model.predict(X_train)
    test_preds = model.predict(X_test)
    test_probs = model.predict_proba(X_test)[:,1]

    seed_baseline_acc = y_test.mean()

    train_acc = accuracy_score(y_train, train_preds)
    test_acc = accuracy_score(y_test, test_preds)
    auc = roc_auc_score(y_test, test_probs)

    print(f'\n  Seed-only baseline accuracy: {seed_baseline_acc:.3f}')
    print(f'  Train accuracy: {train_acc:.3f}')
    print(f'  Test accuracy: {test_acc:.3f} <- compare to baseline')
    print(f'  Test ROC-AUC: {auc:.3f} (0.5 = random, 1.0 = perfect)')
 
    overfit_gap = train_acc - test_acc
    if overfit_gap > 0.08:
        print(f'\n  [WARN]  Overfit gap = {overfit_gap:.3f} — consider lowering C')
    else:
        print(f'\n  [OK]  Overfit gap = {overfit_gap:.3f}')

    cm = confusion_matrix(y_test, test_preds)
    print(f'\n  Confusion matrix (test):')
    print(f'                   Pred: Upset   Pred: Higher seed wins')
    print(f'  Actual: Upset        {cm[0,0]:>5}   {cm[0,1]:>5}')
    print(f'  Actual: Higher seed  {cm[1,0]:>5}   {cm[1,1]:>5}')
 
    print(f'\n  Classification report (test):\n')
    print(classification_report(y_test, test_preds,
                                 target_names=['Upset', 'Higher seed wins']))
    

    coef_df = pd.DataFrame({
        'feature':     feature_cols,
        'coefficient': model.coef_[0],
    }).sort_values('coefficient', ascending=False)
 
    print(f'  Feature coefficients (after scaling — comparable magnitudes):')
    print(coef_df.to_string(index=False))
    print()
    print(f'  Interpretation:')
    print(f'    + coefficient → feature favors higher seed winning')
    print(f'    - coefficient → feature favors upset')

 
    test_df = test_df.copy()
    test_df['PRED'] = test_preds
    test_df['CORRECT'] = (test_df['PRED'] == test_df[TARGET_COL]).astype(int)
 
    per_season = test_df.groupby('SEASON')['CORRECT'].agg(['sum', 'count', 'mean'])
    per_season.columns = ['correct', 'total', 'accuracy']
    print(f'\n  Per-season accuracy (test):')
    print(per_season.round(3).to_string())
 
    return coef_df





In [3]:
df = pd.read_csv(DATA_PATH)

before = len(df)
df = df.dropna(subset=FEATURE_COLS)
dropped = before - len(df)
if dropped:
    print(f'[INFO]: Dropped {dropped} rows with null features')


train_df, test_df = season_split(df)
X_train, X_test, y_train, y_test, scaler = scale(train_df, test_df, FEATURE_COLS)
model = train_model(X_train, y_train)

coef_df = evaluate(model, X_train, X_test, y_train, y_test,
                    FEATURE_COLS, train_df, test_df)



  Seed-only baseline accuracy: 0.640
  Train accuracy: 0.692
  Test accuracy: 0.707 <- compare to baseline
  Test ROC-AUC: 0.731 (0.5 = random, 1.0 = perfect)

  [OK]  Overfit gap = -0.015

  Confusion matrix (test):
                   Pred: Upset   Pred: Higher seed wins
  Actual: Upset           18       9
  Actual: Higher seed     13      35

  Classification report (test):

                  precision    recall  f1-score   support

           Upset       0.58      0.67      0.62        27
Higher seed wins       0.80      0.73      0.76        48

        accuracy                           0.71        75
       macro avg       0.69      0.70      0.69        75
    weighted avg       0.72      0.71      0.71        75

  Feature coefficients (after scaling — comparable magnitudes):
   feature  coefficient
W_PCT_DIFF     0.444423
 ORTG_DIFF     0.191001
 PACE_DIFF     0.124847
 DRTG_DIFF    -0.291656
 SEED_DIFF    -0.693281

  Interpretation:
    + coefficient → feature favors highe

As of 30th May, 2026 — The current western conference finals series is 3-3. 

Below shows what the model thinks about the current outcome:
- Thunder vs Spurs 
    - WCF Winner vs Knicks


In [4]:
def run_aligned_bracket_simulation(okc, sas, nyk, scaler, model):
    wcf_raw = pd.DataFrame([{
        'ORTG_DIFF':     okc['ORTG'] - sas['ORTG'],
        'DRTG_DIFF':     okc['DRTG'] - sas['DRTG'],
        'PACE_DIFF':     okc['PACE'] - sas['PACE'],
        'NET_RTG_DIFF':  okc['NET_RTG'] - sas['NET_RTG'],
        'W_PCT_DIFF':    okc['W_PCT'] - sas['W_PCT'],
        'SEED_DIFF':     okc['SEED'] - sas['SEED']
    }])[FEATURE_COLS]
    
    wcf_scaled = scaler.transform(wcf_raw.values)
    wcf_probs = model.predict_proba(wcf_scaled)[0]
    
    print(f'Oklahoma City Thunder Win Probability: {wcf_probs[1]*100:.1f}%')
    print(f'San Antonio Spurs Upset Probability: {wcf_probs[0]*100:.1f}%')
    
    if wcf_probs[1] > wcf_probs[0]:
        west_winner_name = 'OKC'
        west_winner_stats = okc
        print('VERDICT: Oklahoma City Thunder advance to the Finals.\n')
    else:
        west_winner_name = 'SAS'
        west_winner_stats = sas
        print('VERDICT: San Antonio Spurs pull off the upset to advance.\n')
        
    
    if west_winner_stats['W_PCT'] >= nyk['W_PCT']:
        team_a_name, team_a = west_winner_name, west_winner_stats
        team_b_name, team_b = 'NYK', nyk
    else:
        team_a_name, team_a = 'NYK', nyk
        team_b_name, team_b = west_winner_name, west_winner_stats

    finals_raw = pd.DataFrame([{
        'ORTG_DIFF':     team_a['ORTG'] - team_b['ORTG'],
        'DRTG_DIFF':     team_a['DRTG'] - team_b['DRTG'],
        'PACE_DIFF':     team_a['PACE'] - team_b['PACE'],
        'NET_RTG_DIFF':  team_a['NET_RTG'] - team_b['NET_RTG'],
        'W_PCT_DIFF':    team_a['W_PCT'] - team_b['W_PCT'],
        'SEED_DIFF':     team_a['SEED'] - team_b['SEED']
    }])[FEATURE_COLS]
    
    finals_scaled = scaler.transform(finals_raw.values)
    finals_probs = model.predict_proba(finals_scaled)[0]
    
    print(f'Finals Matchup: {team_a_name} vs. {team_b_name}')
    print(f'  {team_a_name} (Home Court Favorite) Win Probability: {finals_probs[1]*100:.1f}%')
    print(f'  {team_b_name} (Underdog) Upset Probability: {finals_probs[0]*100:.1f}%')
    
    champion = team_a_name if finals_probs[1] > finals_probs[0] else team_b_name
    print(f'\n🏆 THE MODEL PROJECTS THE OFFICIAL 2026 NBA CHAMPION: {champion}')

okc_stats_2026 = {'TEAM_ABB': 'OKC', 'SEED': 1, 'W_PCT': 0.780, 'NET_RTG': 7.3, 'ORTG': 118.5, 'DRTG': 111.2, 'PACE': 100.8}
sas_stats_2026 = {'TEAM_ABB': 'SAS', 'SEED': 2, 'W_PCT': 0.756, 'NET_RTG': 6.8, 'ORTG': 116.2, 'DRTG': 109.4, 'PACE': 99.2}
knicks_stats_2026 = {'TEAM_ABB': 'NYK', 'SEED': 3, 'W_PCT': 0.646, 'NET_RTG': 5.2, 'ORTG': 115.4, 'DRTG': 110.2, 'PACE': 98.0}

run_aligned_bracket_simulation(okc_stats_2026, sas_stats_2026, knicks_stats_2026, scaler, model)

Oklahoma City Thunder Win Probability: 28.9%
San Antonio Spurs Upset Probability: 71.1%
VERDICT: San Antonio Spurs pull off the upset to advance.

Finals Matchup: SAS vs. NYK
  SAS (Home Court Favorite) Win Probability: 41.2%
  NYK (Underdog) Upset Probability: 58.8%

🏆 THE MODEL PROJECTS THE OFFICIAL 2026 NBA CHAMPION: NYK


/Users/ksiu/ML Training/NBA-Playoff-Series-Prediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/Users/ksiu/ML Training/NBA-Playoff-Series-Prediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Based on this outcome from the model, I will update the results after the finals are finished.

31/05/2026 — Spurs Won WCF, so the model made the correct prediction, where Spurs upsted Thunder

1/06/2026 - The inclusion of first round data has improved the model metrics, test and train have increased, so has ROC-AUC
-  The model still leans towards predicting higher seed wins rather than upsets
-  THe main improvement here is model accuracy

